# instantnpgで台風のオリジナル画像を学習する

In [3]:
import os
print(os.getcwd())
scene_path = os.getcwd()
if not os.path.isdir(scene_path):
  raise NotADirectoryError(scene_path)

/media/rc/Transcend/大学院資料/修論データ


In [5]:
# # 使用しているメモリを手放す？
# import torch
# torch.cuda.empty_cache()
# import gc
# gc.collect()
# torch.cuda.memory_summary(device=None, abbreviated=False)

In [ ]:
train_steps = 40000 # 学習回数の指定
# snapshot_path = os.path.join(scene_path, f"{train_steps}.ingp")
snapshot_path = os.path.join(scene_path, f"{train_steps}.msgpack")
marching_cubes_res_n = 827 #aabb_scale 2 たぶん最大値 860, aabb_scale 4 最大値 827
save_mesh_path = os.path.join(os.getcwd(), f"nerf_mesh{marching_cubes_res_n}.ply")
!python3 "/home/rc/instant-ngp/scripts/run2.py" --mode nerf --scene {scene_path} --n_steps {train_steps} --save_snapshot {snapshot_path} --save_mesh {save_mesh_path} --marching_cubes_res {marching_cubes_res_n}

学習済みモデルからMeshを作成する

In [45]:
# marching_cubes_res_n = 827 #aabb_scale 2 たぶん最大値 860, aabb_scale 4 最大値 827
# save_mesh_path = f"/media/rc/Earth/Earth2023/instant-ngp/eye/nerf_mesh{marching_cubes_res_n}.ply"
# !python "/home/rc/instant-ngp/scripts/run2.py" --mode nerf --load_snapshot {snapshot_path} --save_mesh {save_mesh_path} --marching_cubes_res {marching_cubes_res_n}

20:29:53 SUCCESS  Initialized CUDA. Active GPU is #0: NVIDIA GeForce RTX 3090 [86]
20:29:53 INFO     Loading network snapshot from: /media/rc/Earth/Earth2023/instant-ngp/eye/35000.msgpack
20:29:53 INFO     GridEncoding:  Nmin=16 b=2.43803 F=4 T=2^19 L=8
20:29:53 INFO     Density model: 3--[HashGrid]-->32--[FullyFusedMLP(neurons=64,layers=3)]-->1
20:29:53 INFO     Color model:   3--[Composite]-->16+16--[FullyFusedMLP(neurons=64,layers=4)]-->3
20:29:53 INFO       total_encoding_params=12855296 total_network_params=10240
Generating mesh via marching cubes and saving to /media/rc/Earth/Earth2023/instant-ngp/eye/nerf_mesh828.ply. Resolution=[827,827,827]
20:29:54 INFO     #vertices=47567983 #triangles=94473300


transforms.jsonをコピーして、evaluation.jsonファイルを作成して学習画像をPSNRで評価する

In [49]:
# evaluation.jsonを作成する

import shutil
import json
import os

os.makedirs(os.path.join(scene_path, "evaluation"), exist_ok=True)
transforms_file_path = os.path.join(scene_path, "transforms.json")
evaluation_file_path = os.path.join(scene_path, "evaluation", "evaluation.json")

shutil.copy2(transforms_file_path, evaluation_file_path)

with open(evaluation_file_path, "r") as read_file:
    data = json.load(read_file)

for frames_num in range(len(data['frames'])):
    data['frames'][frames_num]['file_path'] = "." + data['frames'][frames_num]['file_path']
    # print(data['frames'][frames_num]['file_path'])

with open(evaluation_file_path, "w") as write_file:
    json.dump(data, write_file, indent=4)

In [50]:
frame_number = 1
!python "/home/rc/instant-ngp/scripts/run2.py" --mode nerf --load_snapshot {snapshot_path} --test_transforms {evaluation_file_path} --screenshot_spp {frame_number}

21:24:08 SUCCESS  Initialized CUDA. Active GPU is #0: NVIDIA GeForce RTX 3090 [86]
21:24:08 INFO     Loading network snapshot from: /media/rc/Earth/Earth2023/instant-ngp/eye/40000.msgpack
21:24:08 INFO     GridEncoding:  Nmin=16 b=2.43803 F=4 T=2^19 L=8
21:24:08 INFO     Density model: 3--[HashGrid]-->32--[FullyFusedMLP(neurons=64,layers=3)]-->1
21:24:08 INFO     Color model:   3--[Composite]-->16+16--[FullyFusedMLP(neurons=64,layers=4)]-->3
21:24:08 INFO       total_encoding_params=12855296 total_network_params=10240
Evaluating test transforms from  /media/rc/Earth/Earth2023/instant-ngp/eye/evaluation/evaluation.json
21:24:13 INFO     Loading NeRF dataset from
21:24:13 INFO       /media/rc/Earth/Earth2023/instant-ngp/eye/evaluation/evaluation.json
21:24:13 PROGRESS [                                   ]   0% (   0/1050)  0s/inf21:24:13 PROGRESS [                                   ]   0% (   1/1050)  0s/23s21:24:13 PROGRESS [                                   ]   0% (   2/1050)  0s/15s2

base_cam.jsonファイルをGUIで作成する

In [46]:
!python "/home/rc/instant-ngp/scripts/run2.py" --mode nerf --scene {scene_path} --load_snapshot {snapshot_path} --gui

21:01:41 SUCCESS  Initialized CUDA. Active GPU is #0: NVIDIA GeForce RTX 3090 [86]
21:01:41 INFO     Loading NeRF dataset from
21:01:41 WARNING    /media/rc/Earth/Earth2023/instant-ngp/eye/base_cam.json does not contain any frames. Skipping.
21:01:41 INFO       /media/rc/Earth/Earth2023/instant-ngp/eye/transforms.json
21:01:41 PROGRESS [                                   ]   0% (   0/1050)  0s/inf21:01:41 PROGRESS [                                   ]   0% (   1/1050)  0s/29s21:01:41 PROGRESS [                                   ]   0% (   2/1050)  0s/18s21:01:41 PROGRESS [                                   ]   0% (   3/1050)  0s/14s21:01:41 PROGRESS [                                   ]   0% (   4/1050)  0s/11s21:01:41 PROGRESS [                                   ]   0% (   5/1050)  0s/11s21:01:41 PROGRESS [                                   ]   1% (   6/1050)  0s/11s21:01:41 PROGRESS [                                   ]   1% (   7/1050)  0s/11s21:01:41 PROGRESS [                     

1. instant-ngp v1.0dev画面

Debug visualizationのAdd training views to camera pathを追加

2. Camera path画面

Path manipulationのsaveを押して、base_cam.jsonファイルを保存する


# Render Videoを作る

In [11]:
video_camera_path = os.path.join(scene_path, "base_cam.json")
if not os.path.isfile(video_camera_path):
  raise FileNotFoundError(video_camera_path)

In [12]:
minutes = 0
secondas = 10
video_n_seconds = minutes*60 + secondas
video_fps = 2
width = 1574
height = 900
output_video_path = os.path.join(scene_path, "render_video.mp4")

!python "/home/rc/instant-ngp/scripts/run2.py" {snapshot_path} --video_camera_path {video_camera_path} --video_n_seconds {video_n_seconds} --video_fps {video_fps} --width {width} --height {height} --video_output {output_video_path}
print(f"Generated video saved to:\n{output_video_path}")

20:34:39 SUCCESS  Initialized CUDA. Active GPU is #0: NVIDIA GeForce RTX 3090 [86]
20:34:39 INFO     Loading network snapshot from: /media/rc/Earth/Earth2023/instant-ngp/eye/35000.msgpack
20:34:39 INFO     GridEncoding:  Nmin=16 b=2.20818 F=4 T=2^19 L=8
20:34:39 INFO     Density model: 3--[HashGrid]-->32--[FullyFusedMLP(neurons=64,layers=3)]-->1
20:34:39 INFO     Color model:   3--[Composite]-->16+16--[FullyFusedMLP(neurons=64,layers=4)]-->3
20:34:39 INFO       total_encoding_params=12660928 total_network_params=10240
Rendering video: 100%|██████████████████████| 20/20 [00:24<00:00,  1.21s/frames]
ffmpeg version 4.2.7-0ubuntu0.1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 9 (Ubuntu 9.4.0-1ubuntu1~20.04.1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-avisynth --e

# Dual-DMPでMeshのノイズ除去
(meshの容量が大きすぎてできない)

In [ ]:
# import os

# module_path = "/media/rc/Earth/Earth2023/Dual-DMP/"

# root_path = "/media/rc/Earth/Earth2023/Dual-DMP/datasets/"
# make_obj_dir = "Mindulle"
# make_obj_dir_path = root_path + make_obj_dir

# os.makedirs(make_obj_dir_path, exist_ok=True)

# !python {module_path}preprocess/preprocess.py -i {make_obj_dir_path}
# !python {module_path}main4real.py -i {make_obj_dir_path}